In [1]:
%pip install thespian

Note: you may need to restart the kernel to use updated packages.


In [4]:
import time
from thespian.actors import ActorSystem, Actor


class SimplestActor(Actor):
    """Простой актор, который в будущем станет агентом"""
    def __init__(self):
        super().__init__()
        print('Создан новый актор')
        self.messages = []
        self.child_addresses = []

    def receiveMessage(self, msg, sender):
        print(f'Актор с адресом {self.myAddress} получил {msg} от {sender}')
        self.messages.append(msg)
        for message in self.messages:
            print(f'Актор с адресом {self.myAddress} ранее получал сообщение {message}')
        for child in self.child_addresses:
            print(f'Ретранслируем сообщение дочернему актору с адресом {child}')
            self.send(child, msg)
        if msg == 'CREATE_ACTOR':
            child_actor_address = self.createActor(SimplestActor)
            print(f'Создали нового актора, его адрес - {child_actor_address}')
            self.child_addresses.append(child_actor_address)


actorSystem = ActorSystem()
actorAddress1 = actorSystem.createActor(SimplestActor)
time.sleep(0.5)

actorSystem.tell(actorAddress1, "Первое сообщение")
time.sleep(0.3)
actorSystem.tell(actorAddress1, 132)
time.sleep(0.3)
actorSystem.tell(actorAddress1, [1, 3, 5])
time.sleep(0.3)
actorSystem.tell(actorAddress1, {'key': 'value'})
time.sleep(0.5)

actorSystem.tell(actorAddress1, 'CREATE_ACTOR')
time.sleep(0.5)
actorSystem.tell(actorAddress1, 'RETRANSLATED MESSAGE')
time.sleep(1.0)

actorSystem.shutdown()

Создан новый актор
Актор с адресом ActorAddr-/A~a получил Первое сообщение от ActorAddr-System:ExternalRequester
Актор с адресом ActorAddr-/A~a ранее получал сообщение Первое сообщение
Актор с адресом ActorAddr-/A~a получил 132 от ActorAddr-System:ExternalRequester
Актор с адресом ActorAddr-/A~a ранее получал сообщение Первое сообщение
Актор с адресом ActorAddr-/A~a ранее получал сообщение 132
Актор с адресом ActorAddr-/A~a получил [1, 3, 5] от ActorAddr-System:ExternalRequester
Актор с адресом ActorAddr-/A~a ранее получал сообщение Первое сообщение
Актор с адресом ActorAddr-/A~a ранее получал сообщение 132
Актор с адресом ActorAddr-/A~a ранее получал сообщение [1, 3, 5]
Актор с адресом ActorAddr-/A~a получил {'key': 'value'} от ActorAddr-System:ExternalRequester
Актор с адресом ActorAddr-/A~a ранее получал сообщение Первое сообщение
Актор с адресом ActorAddr-/A~a ранее получал сообщение 132
Актор с адресом ActorAddr-/A~a ранее получал сообщение [1, 3, 5]
Актор с адресом ActorAddr-/A~a

In [5]:
import time
from thespian.actors import ActorSystem, Actor


class NumberActorForSum(Actor):
    def __init__(self):
        super().__init__()
        self.value = 0

    def receiveMessage(self, msg, sender):
        print(f'Актор числа с адресом {self.myAddress} получил {msg} от {sender}')
        if not isinstance(msg, dict):
            print(f'  Ожидался dict, игнорирую: {msg}')
            return
        self.value = msg.get('init_value')
        calculator_address = msg.get('calculator_address')
        self.send(calculator_address, self.value)


class CalculatorActorForSum(Actor):
    def __init__(self):
        super().__init__()
        self.values = []

    def receiveMessage(self, msg, sender):
        self.values.append(msg)
        total_sum = sum(self.values)
        print(f'Актор-калькулятор с адресом {self.myAddress} получил {msg} от {sender}, '
              f'итоговая сумма: {total_sum}')


actorSystem = ActorSystem()
calculator_address = actorSystem.createActor(CalculatorActorForSum)
time.sleep(0.3)
# Три ОТДЕЛЬНЫХ актора-числа (в func.ipynb был один на все три сообщения)
number_1 = actorSystem.createActor(NumberActorForSum)
number_2 = actorSystem.createActor(NumberActorForSum)
number_3 = actorSystem.createActor(NumberActorForSum)
time.sleep(0.5)

actorSystem.tell(number_1, {'init_value': 1, 'calculator_address': calculator_address})
time.sleep(0.5)
actorSystem.tell(number_2, {'init_value': 2, 'calculator_address': calculator_address})
time.sleep(0.5)
actorSystem.tell(number_3, {'init_value': 3, 'calculator_address': calculator_address})
time.sleep(1.5)

actorSystem.shutdown()

Актор числа с адресом ActorAddr-/A~b получил {'init_value': 1, 'calculator_address': <thespian.actors.ActorAddress object at 0x00000264A54C9440>} от ActorAddr-System:ExternalRequester
Актор-калькулятор с адресом ActorAddr-/A~a получил 1 от ActorAddr-/A~b, итоговая сумма: 1
Актор числа с адресом ActorAddr-/A~c получил {'init_value': 2, 'calculator_address': <thespian.actors.ActorAddress object at 0x00000264A54C9440>} от ActorAddr-System:ExternalRequester
Актор-калькулятор с адресом ActorAddr-/A~a получил 2 от ActorAddr-/A~c, итоговая сумма: 3
Актор числа с адресом ActorAddr-/A~d получил {'init_value': 3, 'calculator_address': <thespian.actors.ActorAddress object at 0x00000264A54C9440>} от ActorAddr-System:ExternalRequester
Актор-калькулятор с адресом ActorAddr-/A~a получил 3 от ActorAddr-/A~d, итоговая сумма: 6


In [6]:
import time
import operator
from thespian.actors import ActorSystem, Actor

OPERATIONS = {
    '+': operator.add,
    '-': operator.sub,
    '*': operator.mul,
    '/': operator.truediv,
}


class NumberActor(Actor):
    """Актор-число: хранит значение и пересылает его в калькулятор."""
    def __init__(self):
        super().__init__()
        self.value = None

    def receiveMessage(self, msg, sender):
        if not isinstance(msg, dict) or 'value' not in msg:
            print(f'[NumberActor {self.myAddress}] получил странное сообщение: {msg}, игнорирую')
            return
        self.value = msg['value']
        calculator_address = msg['calculator_address']
        print(f'[NumberActor {self.myAddress}] получил значение {self.value}, '
              f'пересылаю в калькулятор {calculator_address}')
        self.send(calculator_address, {'value': self.value})


class CalculatorActor(Actor):
    """Актор-калькулятор: накапливает числа и оператор, затем вычисляет."""
    def __init__(self):
        super().__init__()
        self.numbers = []
        self.operator = None

    def receiveMessage(self, msg, sender):
        if not isinstance(msg, dict):
            print(f'[CalculatorActor {self.myAddress}] получил не-словарь: {msg}, игнорирую')
            return
        print(f'[CalculatorActor {self.myAddress}] получил сообщение: {msg}')
        if 'value' in msg:
            self.numbers.append(msg['value'])
        elif 'operator' in msg:
            self.operator = msg['operator']
        else:
            print('  Неизвестный тип сообщения, игнорирую.')
            return
        if self.operator is not None and len(self.numbers) >= 2:
            a = self.numbers[-2]
            b = self.numbers[-1]
            op = self.operator
            if op in OPERATIONS:
                if op == '/' and b == 0:
                    result = 'Ошибка: деление на ноль'
                else:
                    result = OPERATIONS[op](a, b)
            else:
                result = f'Неизвестная операция: {op}'
            print(f'Выражение: {a} {op} {b} = {result}')
            self.numbers = []
            self.operator = None


actorSystem = ActorSystem('simpleSystemBase')
calculator = actorSystem.createActor(CalculatorActor)
number_agent_1 = actorSystem.createActor(NumberActor)
number_agent_2 = actorSystem.createActor(NumberActor)
time.sleep(0.5)

print('\n=== Тест 1: 2 * 5 ===')
actorSystem.tell(number_agent_1, {'value': 2, 'calculator_address': calculator})
actorSystem.tell(number_agent_2, {'value': 5, 'calculator_address': calculator})
actorSystem.tell(calculator, {'operator': '*'})
time.sleep(1)

print('\n=== Тест 2: 10 + 7 ===')
actorSystem.tell(calculator, {'operator': '+'})
actorSystem.tell(number_agent_1, {'value': 10, 'calculator_address': calculator})
actorSystem.tell(number_agent_2, {'value': 7, 'calculator_address': calculator})
time.sleep(1)

print('\n=== Тест 3: 20 - 8 ===')
actorSystem.tell(number_agent_1, {'value': 20, 'calculator_address': calculator})
actorSystem.tell(calculator, {'operator': '-'})
actorSystem.tell(number_agent_2, {'value': 8, 'calculator_address': calculator})
time.sleep(1)

print('\n=== Тест 4: 15 / 3 ===')
actorSystem.tell(number_agent_1, {'value': 15, 'calculator_address': calculator})
actorSystem.tell(number_agent_2, {'value': 3, 'calculator_address': calculator})
actorSystem.tell(calculator, {'operator': '/'})
time.sleep(1)

actorSystem.shutdown()


=== Тест 1: 2 * 5 ===
[NumberActor ActorAddr-/A~b] получил значение 2, пересылаю в калькулятор ActorAddr-/A~a
[CalculatorActor ActorAddr-/A~a] получил сообщение: {'value': 2}
[NumberActor ActorAddr-/A~c] получил значение 5, пересылаю в калькулятор ActorAddr-/A~a
[CalculatorActor ActorAddr-/A~a] получил сообщение: {'value': 5}
[CalculatorActor ActorAddr-/A~a] получил сообщение: {'operator': '*'}
Выражение: 2 * 5 = 10

=== Тест 2: 10 + 7 ===
[CalculatorActor ActorAddr-/A~a] получил сообщение: {'operator': '+'}
[NumberActor ActorAddr-/A~b] получил значение 10, пересылаю в калькулятор ActorAddr-/A~a
[CalculatorActor ActorAddr-/A~a] получил сообщение: {'value': 10}
[NumberActor ActorAddr-/A~c] получил значение 7, пересылаю в калькулятор ActorAddr-/A~a
[CalculatorActor ActorAddr-/A~a] получил сообщение: {'value': 7}
Выражение: 10 + 7 = 17

=== Тест 3: 20 - 8 ===
[NumberActor ActorAddr-/A~b] получил значение 20, пересылаю в калькулятор ActorAddr-/A~a
[CalculatorActor ActorAddr-/A~a] получил 

In [7]:
from thespian.actors import ActorSystem, Actor
import time


class TimerActor(Actor):
    """Актор-таймер со счётчиком."""
    def __init__(self):
        super().__init__()
        print('Создан новый таймер актор')
        self.counter = 0

    def receiveMessage(self, msg, sender):
        self.counter += 1
        print(f'Таймер актор {self.myAddress} получил "{msg}", счётчик = {self.counter}')
        self.send(sender, self.counter)


actorSystem = ActorSystem()
timerAddress1 = actorSystem.createActor(TimerActor)
timerAddress2 = actorSystem.createActor(TimerActor)
time.sleep(0.5)

for i in range(1, 4):
    resp = actorSystem.ask(timerAddress1, f'привет {i}')
    print(f'   Ответ от timerAddress1: {resp}')

for i in range(1, 3):
    resp = actorSystem.ask(timerAddress2, f'здравствуй {i}')
    print(f'   Ответ от timerAddress2: {resp}')

resp = actorSystem.ask(timerAddress1, 'привет 4', timeout=1)
print(f'   Ответ от timerAddress1 (привет 4): {resp}')

actorSystem.shutdown()

Создан новый таймер актор
Создан новый таймер актор
Таймер актор ActorAddr-/A~a получил "привет 1", счётчик = 1
   Ответ от timerAddress1: 1
Таймер актор ActorAddr-/A~a получил "привет 2", счётчик = 2
   Ответ от timerAddress1: 2
Таймер актор ActorAddr-/A~a получил "привет 3", счётчик = 3
   Ответ от timerAddress1: 3
Таймер актор ActorAddr-/A~b получил "здравствуй 1", счётчик = 1
   Ответ от timerAddress2: 1
Таймер актор ActorAddr-/A~b получил "здравствуй 2", счётчик = 2
   Ответ от timerAddress2: 2
Таймер актор ActorAddr-/A~a получил "привет 4", счётчик = 4
   Ответ от timerAddress1 (привет 4): 4


In [9]:
from thespian.actors import ActorSystem, Actor
import time


class ChainActor(Actor):
    def __init__(self):
        super().__init__()
        self.next_actor = None
        self.is_last = False
        self.is_first = False
        self.first_actor = None

    def receiveMessage(self, msg, sender):
        if not isinstance(msg, dict) and not isinstance(msg, str):
            return
        if isinstance(msg, dict) and 'next' in msg:
            self.next_actor = msg['next']
            self.is_last = msg.get('is_last', False)
            self.is_first = msg.get('is_first', False)
            self.first_actor = msg.get('first_actor')
            return
        if self.is_first and isinstance(msg, dict) and 'result' in msg:
            elapsed = time.perf_counter() - self.start_time
            print(f'Результат: {msg["result"]}, переходов: {msg["hops"]}, '
                  f'время: {elapsed*1000:.2f} мс')
            return
        if msg == 'start':
            self.start_time = time.perf_counter()
            self.send(self.next_actor, {'data': 'Hello', 'hops': 1})
            return
        if not self.is_last:
            self.send(self.next_actor, {'data': msg['data'], 'hops': msg['hops'] + 1})
            return
        if self.is_last:
            self.send(self.first_actor, {'result': msg['data'], 'hops': msg['hops'] + 1})
            return


actorSystem = ActorSystem('simpleSystemBase')
actors = [actorSystem.createActor(ChainActor) for _ in range(5)]
time.sleep(0.5)
for i in range(5):
    next_idx = (i + 1) % 5
    actorSystem.tell(actors[i], {
        'next': actors[next_idx],
        'is_last': i == 4,
        'is_first': i == 0,
        'first_actor': actors[0]
    })
time.sleep(1.0)
actorSystem.tell(actors[0], 'start')
time.sleep(1.5)
actorSystem.shutdown()

Результат: Hello, переходов: 5, время: 0.04 мс
